# Phase 2.2: LSTM Tensor Preparation

## Objectif
Préparer tenseurs LSTM **4 h** (MVP) — aligné KPI horaires et `TRAINING_PLAN.md`.

## Spécifications
- **Cadence**: 1 h | **Lookback**: 48 h | **Horizon**: **4 h** uniquement
- **Split temporel par site** (70/15/15), puis fusion — évite mélanger les chronologies
- **Winsorization** (p1–p99 sur train) puis **MinMax** fit **train only**
- **8 polluants** + **4 features calendrier** (Sprint 3) si `use_calendar_features`

## Sorties
- `ia/data/lstm_train_val_test.pkl` (clé `combined_tensors[4]`)
- `ia/data/lstm_metadata.json` | `ia/models/lstm_scalers.pkl`

In [1]:
import json
import pickle
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
np.random.seed(42)

PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Recharger config si le noyau garde une ancienne version (sans horizons_hourly)
import importlib
import config as config_module
import lstm_training as lstm_training_module

importlib.reload(config_module)
importlib.reload(lstm_training_module)

from config import (
    LSTM_CALENDAR_FEATURE_NAMES,
    LSTM_CONFIG,
    LSTM_FEATURE_NAMES,
    N_OUTPUT_FEATURES,
    lstm_input_feature_count,
)
from lstm_training import (
    build_windows,
    concat_splits_with_test_sites,
    fit_winsor_bounds,
    scale_splits,
    temporal_split,
    winsorize_X_pollutants,
    winsorize_array,
)

N_FEATURES = N_OUTPUT_FEATURES
N_INPUT_FEATURES = lstm_input_feature_count()
USE_CALENDAR = LSTM_CONFIG.get("use_calendar_features", False)

print("config:", (PROJECT_ROOT / "config.py").resolve())
print("lstm_training:", lstm_training_module.__file__)

POLLUTANT_NAMES = LSTM_FEATURE_NAMES
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

LOOKBACK = LSTM_CONFIG["lookback"]
HORIZON = LSTM_CONFIG["horizon_short"]
HORIZONS = LSTM_CONFIG.get("horizons_hourly", [HORIZON])
if HORIZONS != [HORIZON]:
    print(f"Note: horizons_hourly={HORIZONS} — MVP = [{HORIZON}] h seulement")
    HORIZONS = [HORIZON]
TIMESTEP_MINUTES = LSTM_CONFIG["timestep_minutes"]
WINSOR_LO = LSTM_CONFIG.get("winsorize_lower_pct", 1.0)
WINSOR_HI = LSTM_CONFIG.get("winsorize_upper_pct", 99.0)

POLLUTANT_ALIASES = {
    "NO2": "NOX", "NOx": "NOX", "NOX": "NOX",
    "SO2": "SOX", "SOX": "SOX",
    "PM2.5": "PM25", "PM25": "PM25", "PM10": "PM10", "PM1": "PM25",
    "CO2": "CO2", "COV": "COV", "VOC": "COV",
    "TEMPERATURE": "TEMPERATURE", "TEMP": "TEMPERATURE", "T": "TEMPERATURE",
    "HUMIDITY": "HUMIDITY", "RH": "HUMIDITY",
}

print(f"Lookback={LOOKBACK}h, Horizons={HORIZONS}h, Timestep={TIMESTEP_MINUTES}min")

config: C:\Users\melik\Desktop\pollution_monitoring\ia\config.py
lstm_training: C:\Users\melik\Desktop\pollution_monitoring\ia\lstm_training.py
Lookback=48h, Horizons=[4]h, Timestep=60min


In [2]:
def load_hourly_wide(csv_path: Path) -> pd.DataFrame:
    """Long format → pivot → rééchantillonnage 1 h par site."""
    df = pd.read_csv(csv_path)
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
    df["pollutant"] = df["pollutant"].astype(str).str.upper().replace(POLLUTANT_ALIASES)

    site_col = "site_id" if "site_id" in df.columns else None
    if site_col is None:
        df["site_id"] = "default"
        site_col = "site_id"

    frames = []
    for site_id, site_df in df.groupby(site_col):
        wide = site_df.pivot_table(
            index="timestamp_utc", columns="pollutant", values="value", aggfunc="mean"
        )
        wide = wide.resample("1h").mean()

        # Météo : colonnes latérales du CSV (pas dans pivot polluant)
        if "temperature_c" in site_df.columns:
            meteo = site_df.groupby("timestamp_utc").agg(
                temperature_c=("temperature_c", "mean"),
                humidity_percent=("humidity_percent", "mean"),
            )
            meteo = meteo.resample("1h").mean()
            wide["TEMPERATURE"] = meteo["temperature_c"]
            wide["HUMIDITY"] = meteo["humidity_percent"]

        for col in POLLUTANT_NAMES:
            if col not in wide.columns:
                wide[col] = np.nan
        wide = wide[POLLUTANT_NAMES]
        wide = wide.interpolate(method="linear", limit=6).ffill().bfill()
        for col in POLLUTANT_NAMES:
            med = wide[col].median()
            wide[col] = wide[col].fillna(med if pd.notna(med) else 0.0)
        wide["site_id"] = str(site_id)
        frames.append(wide.reset_index())

    return pd.concat(frames, ignore_index=True)


# build_windows : ia/lstm_training.py (polluants + calendrier optionnel)



In [3]:
input_file = DATA_DIR / "training_dataset.csv"
output_file = DATA_DIR / "lstm_train_val_test.pkl"
metadata_file = DATA_DIR / "lstm_metadata.json"
scalers_file = MODELS_DIR / "lstm_scalers.pkl"

print(f"Chargement: {input_file}")
hourly_df = load_hourly_wide(input_file)
print(f"Lignes horaires: {len(hourly_df):,} | Sites: {hourly_df['site_id'].nunique()}")

Chargement: C:\Users\melik\Desktop\pollution_monitoring\ia\data\training_dataset.csv
Lignes horaires: 179,417 | Sites: 25


In [4]:
scalers = {}
combined_tensors = {}
window_counts = {}
train_ratio = LSTM_CONFIG["train_ratio"]
val_ratio = LSTM_CONFIG["val_ratio"]

for horizon in HORIZONS:
    splits_per_site = []
    site_ids_order = []
    window_counts[horizon] = {}

    for site_id, site_df in hourly_df.groupby("site_id"):
        site_df = site_df.sort_values("timestamp_utc")
        matrix = site_df[POLLUTANT_NAMES].values.astype(np.float32)
        ts = site_df["timestamp_utc"].values if USE_CALENDAR else None
        X_s, y_s = build_windows(
            matrix, LOOKBACK, horizon, timestamps=ts, n_pollutant_features=N_OUTPUT_FEATURES
        )
        if len(X_s) < 100:
            continue
        site_ids_order.append(str(site_id))
        splits_per_site.append(
            temporal_split(X_s, y_s, train_ratio, val_ratio)
        )
        window_counts[horizon][site_id] = len(X_s)

    if not splits_per_site:
        raise ValueError(f"Pas assez de fenêtres pour horizon={horizon}h")

    (X_train, y_train), (X_val, y_val), (X_test, y_test), test_site_ids = (
        concat_splits_with_test_sites(splits_per_site, site_ids_order)
    )

    bounds = fit_winsor_bounds(
        [X_train[:, :, :N_OUTPUT_FEATURES], y_train],
        N_OUTPUT_FEATURES,
        lower_q=WINSOR_LO,
        upper_q=WINSOR_HI,
    )
    X_train = winsorize_X_pollutants(X_train, *bounds, N_OUTPUT_FEATURES)
    y_train = winsorize_array(y_train, *bounds, N_OUTPUT_FEATURES)
    X_val = winsorize_X_pollutants(X_val, *bounds, N_OUTPUT_FEATURES)
    y_val = winsorize_array(y_val, *bounds, N_OUTPUT_FEATURES)
    X_test = winsorize_X_pollutants(X_test, *bounds, N_OUTPUT_FEATURES)
    y_test = winsorize_array(y_test, *bounds, N_OUTPUT_FEATURES)

    X_train, y_train, X_val, y_val, X_test, y_test, scaler = scale_splits(
        X_train, y_train, X_val, y_val, X_test, y_test, N_OUTPUT_FEATURES
    )
    scalers[horizon] = scaler
    combined_tensors[horizon] = {
        "train": (X_train, y_train),
        "val": (X_val, y_val),
        "test": (X_test, y_test),
        "test_site_ids": test_site_ids,
    }
    print(
        f"Horizon {horizon}h: train={X_train.shape}, "
        f"nan={np.isnan(X_train).sum()}, sites={len(window_counts[horizon])}"
    )

joblib.dump({"scaler": scalers[HORIZON], "winsor_bounds": bounds}, scalers_file)
print(f"Scaler + winsor: {scalers_file}")

Horizon 4h: train=(124711, 48, 8), nan=0, sites=22
Scaler + winsor: C:\Users\melik\Desktop\pollution_monitoring\ia\models\lstm_scalers.pkl


In [5]:
metadata = {
    "timestep_minutes": TIMESTEP_MINUTES,
    "lookback": LOOKBACK,
    "lookback_hours": LOOKBACK,
    "horizons": HORIZONS,
    "horizon_hours": {str(h): h for h in HORIZONS},
    "pollutant_names": POLLUTANT_NAMES,
    "n_output_features": N_OUTPUT_FEATURES,
    "n_input_features": N_INPUT_FEATURES,
    "n_features": N_OUTPUT_FEATURES,
    "use_calendar_features": USE_CALENDAR,
    "calendar_feature_names": list(LSTM_CALENDAR_FEATURE_NAMES) if USE_CALENDAR else [],
    "window_counts": window_counts,
    "split_strategy": "temporal_per_site",
    "winsorize_pct": [WINSOR_LO, WINSOR_HI],
}

lstm_data = {
    "combined_tensors": combined_tensors,
    "scalers": scalers,
    "lookback": LOOKBACK,
    "horizons": HORIZONS,
    "timestep_minutes": TIMESTEP_MINUTES,
    "pollutant_names": POLLUTANT_NAMES,
    "metadata": metadata,
}

with open(output_file, "wb") as f:
    pickle.dump(lstm_data, f)

with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Tenseurs: {output_file} ({output_file.stat().st_size / 1024 / 1024:.2f} MB)")
print(f"Metadata: {metadata_file}")

Tenseurs: C:\Users\melik\Desktop\pollution_monitoring\ia\data\lstm_train_val_test.pkl (283.97 MB)
Metadata: C:\Users\melik\Desktop\pollution_monitoring\ia\data\lstm_metadata.json


## Résumé

- Horizon **4 h** seul | split **par site** | winsor p1–p99 | MinMax sur train
- Prochaine étape: notebook **06** (`model_lstm_4h.h5`)